# Session 3: RepWise's MCP server

This notebook calls RepWise's own [`server.py`](../server.py) over the Model Context Protocol. The server exposes three tools, which use the same pydantic schemas and database code as the RepWise API.

It needs no API key and touches no real data: it creates a temporary database with one synthetic account. To run it, use **Run all**, in Colab or from a local checkout.

In [1]:
#@title Connect to RepWise's server.py
# Works in Colab (clones the public repository) and from a local checkout.
import asyncio, json, os, subprocess, sys, tempfile
from pathlib import Path

try:
    import dotenv, fastmcp
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastmcp", "python-dotenv"], check=True)
from fastmcp import Client

WORK = tempfile.mkdtemp(prefix="repwise-s3-")
try:
    sys.stderr.fileno()     # The MCP stdio client needs a real file for server logs.
except Exception:
    sys.stderr = open(os.path.join(WORK, "mcp_servers.log"), "a")

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "server.py").exists() and (p / "agentic").exists()), None)
if REPO is None:
    if not Path("RepWise").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Kingflow-23/RepWise.git"], check=True)
    REPO = Path("RepWise").resolve()

# A temporary database with one synthetic account, so no real data is used.
DB = os.path.join(WORK, "repwise_demo.sqlite3")
seed = subprocess.run(
    [sys.executable, "-c",
     "import sys; from pathlib import Path; from agentic.mcp_demo_client import seed_demo_account; "
     "print(','.join(seed_demo_account(Path(sys.argv[1]))))", DB],
    cwd=REPO, capture_output=True, text=True, check=True)
STUDENT_ID, PLAN_ID = seed.stdout.strip().splitlines()[-1].split(",")

REPWISE_SERVER = {"command": sys.executable, "args": ["server.py"], "cwd": str(REPO),
                  "env": {"FASTMCP_LOG_LEVEL": "WARNING", "REPWISE_DATABASE_PATH": DB, "REPWISE_LOG_DIR": WORK}}
REPWISE_CONFIG = {"mcpServers": {"repwise": REPWISE_SERVER}}

def output_of(result):
    """Return a tool's validated output as a dictionary."""
    return result.structured_content["result"]

print("Server: server.py")
print("Database: temporary, one synthetic account")

Server: server.py
Database: temporary, one synthetic account


## Discover the tools

The client asks the server what it offers. FastMCP builds the schemas from the Python type hints, so `difficulty` becomes an enum.

In [2]:
#@title Discover the tools
async with Client(REPWISE_CONFIG) as mc:
    tools = await mc.list_tools()
for t in tools:
    print(f"{t.name}({', '.join(t.input_schema['properties'])})\n   {t.description}\n")
difficulty = next(t for t in tools if t.name == "save_workout_feedback").input_schema["properties"]["difficulty"]
print("difficulty enum:", difficulty["enum"])

get_user_profile(student_id)
   Read a student's experience, goal, available_days, session_minutes, equipment, and preferences.

get_timetable(student_id)
   Read a student's weekly availability and planned workouts, each with its plan_id, week_number, workout_index, and saved feedback.

save_workout_feedback(student_id, plan_id, week_number, workout_index, completion_status, difficulty, comment)
   Save feedback for one workout. Call only when the student explicitly asks to log it.

difficulty enum: ['much_too_easy', 'slightly_too_easy', 'just_right', 'slightly_too_hard', 'much_too_hard']


## Call the tools

This is what a host does when a model asks for a call. The timetable supplies the values the save needs, invalid input is rejected, and a new server process reloads the saved feedback.

In [3]:
#@title Call all three tools, without an LLM
async with Client(REPWISE_CONFIG) as mc:
    profile = output_of(await mc.call_tool("get_user_profile", {"student_id": STUDENT_ID}))
    print("get_user_profile →", json.dumps(profile["profile"], indent=1))

    timetable = output_of(await mc.call_tool("get_timetable", {"student_id": STUDENT_ID}))
    print("\nget_timetable →", timetable["available_days"], timetable["session_minutes"], "minutes")
    for w in timetable["planned_workouts"]:
        print(f"   {w['day']}: {w['focus']} (week {w['week_number']}, workout_index {w['workout_index']})")

    first = timetable["planned_workouts"][0]
    saved = output_of(await mc.call_tool("save_workout_feedback", {
        "student_id": STUDENT_ID, "plan_id": first["plan_id"], "week_number": first["week_number"],
        "workout_index": first["workout_index"], "completion_status": "completed",
        "difficulty": "slightly_too_hard", "comment": "Called from the Session 3 notebook"}))
    print("\nsave_workout_feedback → saved:", saved["saved"], "|", saved["feedback"]["difficulty"], "|", saved["feedback"]["comment"])

    # Invalid input is rejected, by our tool or by the server's schema validation.
    print("\nUnknown student →", output_of(await mc.call_tool("get_user_profile", {"student_id": "unknown"})))
    try:
        await mc.call_tool("save_workout_feedback", {
            "student_id": STUDENT_ID, "plan_id": first["plan_id"], "week_number": 1, "workout_index": 0,
            "completion_status": "completed", "difficulty": "hard"})
    except Exception as error:
        print("Invalid difficulty → rejected by the server:", str(error).splitlines()[0])

# A new server process only sees this record if it reached the database.
async with Client(REPWISE_CONFIG) as mc:
    reloaded = output_of(await mc.call_tool("get_timetable", {"student_id": STUDENT_ID}))
print("\nReloaded by a new server process:", reloaded["planned_workouts"][0]["feedback"]["comment"])

get_user_profile → {
 "experience": "beginner",
 "goal": "Build general strength",
 "available_days": [
  "Monday",
  "Wednesday",
  "Friday"
 ],
 "session_minutes": 45,
 "equipment": [
  "dumbbells",
  "bodyweight"
 ],
 "preferences": "Prefers short rests and simple movements."
}

get_timetable → ['Monday', 'Wednesday', 'Friday'] 45 minutes
   Monday: Lower body strength (week 1, workout_index 0)
   Wednesday: Upper body strength (week 1, workout_index 1)
   Friday: Full body basics (week 1, workout_index 2)

save_workout_feedback → saved: True | slightly_too_hard | Called from the Session 3 notebook

Unknown student → {'ok': False, 'code': 'student_not_found', 'error': 'Student profile not found'}
Invalid difficulty → rejected by the server: 1 validation error for call[save_workout_feedback]



Reloaded by a new server process: Called from the Session 3 notebook
